# 🎤 RVC v2: Персональный голос для песен (Сергей Трофимов — Аты-баты)

**Инструкция: Нажмите `Среда выполнения` -> `Выполнить все` (Runtime -> Run all).**

Пайплайн выполнится полностью автоматически за **~10–12 минут**:
1. Разворачивается официальная среда RVC v2 (Applio) на GPU Tesla T4.
2. Подтягивается ваш чистый 9-минутный голос (`Testr2.wav`) и трек `Сергей Трофимов - Аты-баты`.
3. Извлекаются частоты связок (алгоритм RMVPE) и обучается модель (150 эпох, ~7-8 мин).
4. Нейросеть Demucs разделяет песню на музыку и вокал.
5. RVC заменяет оригинальный голос Трофимова на **ваш персональный голос** с сохранением всех нот и интонаций!
6. Готовый трек проигрывается прямо в блокноте и скачивается на устройство.

## Шаг 1. Проверка GPU и установка официального окружения Applio (RVC v2)

In [ ]:
!nvidia-smi

import os
%cd /content
!git config --global advice.detachedHead false
if not os.path.exists("/content/Applio"):
    !git clone --depth 1 https://github.com/IAHispano/Applio.git /content/Applio
%cd /content/Applio

!apt-get update -y -qq
!apt-get install -y -qq portaudio19-dev ffmpeg git curl

print("Установка uv и окружения Applio (Python 3.12, CUDA 12.8 / PyTorch)...\n")
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv venv .venv --python 3.12 --seed
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --python /content/Applio/.venv/bin/python
!uv pip install -q matplotlib-inline websockets --python /content/Applio/.venv/bin/python
!/content/Applio/.venv/bin/python core.py prerequisites --models --pretraineds-hifigan --exe
!cp assets/config_template.json assets/config.json

print("Установка Demucs для выделения вокала...")
!pip install -q demucs

# Проверка работоспособности
!/content/Applio/.venv/bin/python -c "import torch; print(\"CUDA in Applio venv:\", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"No GPU\")"
!python3 -c "import demucs; print(\"Demucs готов!\")"
print("Шаг 1 завершен успешно!")

## Шаг 2. Загрузка вашего чистого голоса и целевого трека

In [ ]:
import os, shutil

%cd /content
if not os.path.exists("/content/piper-voice"):
    !git clone --depth 1 https://github.com/Holki9/piper-voice.git /content/piper-voice
else:
    !cd /content/piper-voice && git pull origin main

dataset_dir = "/content/Applio/assets/datasets/MyVoice"
os.makedirs(dataset_dir, exist_ok=True)
shutil.copy2("/content/piper-voice/raw_audio/Testr2.wav", os.path.join(dataset_dir, "Testr2.wav"))
shutil.copy2("/content/piper-voice/songs/aty_baty.mp3", "/content/aty_baty.mp3")

print("Файлы проверены и скопированы:")
!ls -lh /content/Applio/assets/datasets/MyVoice/
!ls -lh /content/aty_baty.mp3

## Шаг 3. Предобработка и извлечение частот связок (RMVPE)

In [ ]:
%cd /content/Applio

# 1. Нарезка и ресемплинг
!/content/Applio/.venv/bin/python core.py preprocess --model-name MyVoice --dataset-path assets/datasets/MyVoice --sample-rate 40000 --cpu-cores 2 --clean

# 2. Извлечение питча (RMVPE) и контент-эмбеддингов (ContentVec) на GPU
!/content/Applio/.venv/bin/python core.py extract --model-name MyVoice --sample-rate 40000 --f0-method rmvpe --embedder-model contentvec

print("Шаг 3 завершен успешно!")

## Шаг 4. Обучение модели RVC v2 на вашем голосе (~7-8 минут на GPU T4)

In [ ]:
%cd /content/Applio

# Запуск быстрого обучения на GPU T4 (~7-8 минут на 150 эпох)
!/content/Applio/.venv/bin/python core.py train --model-name MyVoice --save-every-epoch 25 --total-epoch 150 --sample-rate 40000 --batch-size 4 --gpu 0 --vocoder HiFi-GAN --save-only-latest --pretrained

# Построение индекса поиска тембра
!/content/Applio/.venv/bin/python core.py index --model-name MyVoice --index-algorithm Auto

print("Шаг 4 завершен! Обученная модель:")
!ls -lh logs/MyVoice/

## Шаг 5. Спеть песню твоим голосом и скачать результат

In [ ]:
import glob, os, subprocess, shlex
from IPython.display import Audio, display
from google.colab import files

print("1. Разделение трека на вокал и инструментал (Demucs)...")
!demucs --two-stems=vocals /content/aty_baty.mp3 -o /content/separated

vocal_track = "/content/separated/htdemucs/aty_baty/vocals.wav"
no_vocal_track = "/content/separated/htdemucs/aty_baty/no_vocals.wav"

if not os.path.exists(vocal_track):
    vocals = glob.glob("/content/separated/**/vocals.wav", recursive=True)
    if vocals:
        vocal_track = vocals[0]
        no_vocal_track = vocal_track.replace("vocals.wav", "no_vocals.wav")

# Поиск путей обученной модели и индекса
!ls -t /content/Applio/logs/MyVoice/*.pth 2> /dev/null | head -n 1 > /tmp/pth.txt
pth_file = open("/tmp/pth.txt").read().strip()
if not pth_file:
    raise FileNotFoundError("Не найден файл обученной модели .pth")

!ls -t /content/Applio/logs/MyVoice/*.index 2> /dev/null | head -n 1 > /tmp/index.txt
index_file = open("/tmp/index.txt").read().strip()

print(f"Используем модель: {pth_file}")
print(f"Используем индекс: {index_file}")

print("2. Перепеваем вокальную дорожку твоим голосом (RVC v2)...\n")
%cd /content/Applio
infer_cmd = [
    "/content/Applio/.venv/bin/python", "core.py", "infer",
    "--pitch", "0",
    "--volume-envelope", "0.8",
    "--index-rate", "0.75",
    "--protect", "0.5",
    "--f0-method", "rmvpe",
    "--input-path", vocal_track,
    "--output-path", "/content/my_vocals.wav",
    "--pth-path", pth_file,
    "--export-format", "WAV",
    "--embedder-model", "contentvec"
]
if index_file and os.path.exists(index_file):
    infer_cmd += ["--index-path", index_file]

!{shlex.join(infer_cmd)}

print("3. Сведение твоего вокала с оригинальной аранжировкой...")
mix_cmd = [
    "ffmpeg", "-y",
    "-i", "/content/my_vocals.wav",
    "-i", no_vocal_track,
    "-filter_complex", "[0:a]volume=1.15[v];[1:a]volume=0.95[m];[v][m]amix=inputs=2:duration=first:dropout_transition=2",
    "-b:a", "320k",
    "/content/Трофимов_Аты_баты_Мой_голос.mp3"
]
subprocess.run(mix_cmd, check=True)

print("\n🎉 ГОТОВО! Песня Сергей Трофимов — Аты-баты перепета твоим голосом!")
display(Audio("/content/Трофимов_Аты_баты_Мой_голос.mp3"))

# Скачивание готовой песни и модели на устройство
print("Скачивание готового трека...")
files.download("/content/Трофимов_Аты_баты_Мой_голос.mp3")
if pth_file and os.path.exists(pth_file):
    files.download(pth_file)
if index_file and os.path.exists(index_file):
    files.download(index_file)
